# HGB tree baseline for logistics (multi-output regression, no new deps)
Adds EXACTLY one model to DNN/ANN-DCN comparison: `HistGradientBoostingRegressor` (sklearn-native XGBoost analogue; `xgboost` absent, installs forbidden).
Same 7 inputs / 17 targets / `SEED=42` 32k-fit/8k-val/10k-test split / `calculate_metrics()` as `compare_logistics_deep_learning.ipynb`. Trains one estimator per target.
Run from `backend/model/`. Writes only `model_logistics_hgb.pkl`, `logistics_hgb_comparison.csv`, `logistics_hgb_per_item_mae.csv`.


In [1]:
import gc, pickle
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
SEED = 42; np.random.seed(SEED)
FEATURES = ['damage_count','affected_kk','total_population','disaster_type_enc','severity_level','emergency_duration','vulnerability_idx']
TARGETS = ['beras_kg','minyak_liter','gula_kg','indomie_pcs','roma_sari_gandum_pack','roma_malkist_abon_pack','roma_kelapa_pack','roma_marie_susu_pack','sarden_pcs','kornet_pcs','susu_full_cream_pcs','susu_dancow_box','matras_pcs','kasur_lipat_pcs','kompor_set','karpet_plastik_pcs','kipas_angin_pcs']

In [2]:
# Identical synthetic generator + identical construction order (DNN/DCN/MLP protocol)
n_samples = 50000
damage_count = np.random.randint(1, 200, size=n_samples)
affected_kk = np.maximum(1, np.round(damage_count * np.random.uniform(0.9, 1.2, size=n_samples))).astype(int)
total_population = affected_kk * np.random.randint(3, 6, size=n_samples)
disaster_type_enc = np.random.randint(0, 6, size=n_samples)
severity_level = np.random.randint(1, 6, size=n_samples)
emergency_duration = np.random.choice([3, 7, 10, 14], size=n_samples)
vulnerability_idx = np.random.uniform(0.1, 1.0, size=n_samples)
duration_factor = emergency_duration / 10.0
vuln_factor = 0.9 + (vulnerability_idx * 0.2)
def scaled(base): return base * duration_factor * vuln_factor * np.random.uniform(0.97, 1.03, size=n_samples)
x = pd.DataFrame({'damage_count': damage_count, 'affected_kk': affected_kk, 'total_population': total_population,
 'disaster_type_enc': disaster_type_enc, 'severity_level': severity_level,
 'emergency_duration': emergency_duration, 'vulnerability_idx': vulnerability_idx})[FEATURES]
y = pd.DataFrame({
 'beras_kg': np.round(scaled(affected_kk*5.0),1), 'minyak_liter': np.round(scaled(affected_kk*1.0),1),
 'gula_kg': np.round(scaled(affected_kk*1.0),1), 'indomie_pcs': np.round(scaled(affected_kk*5.0)).astype(int),
 'roma_sari_gandum_pack': np.round(scaled(affected_kk*1.0)).astype(int),
 'roma_malkist_abon_pack': np.round(scaled(affected_kk*1.0)).astype(int),
 'roma_kelapa_pack': np.round(scaled(affected_kk*1.0)).astype(int),
 'roma_marie_susu_pack': np.round(scaled(affected_kk*1.0)).astype(int),
 'sarden_pcs': np.round(scaled(affected_kk*1.0)).astype(int), 'kornet_pcs': np.round(scaled(affected_kk*1.0)).astype(int),
 'susu_full_cream_pcs': np.round(scaled(affected_kk*1.0)).astype(int),
 'susu_dancow_box': np.round(scaled(affected_kk*1.0)).astype(int),
 'matras_pcs': np.maximum(1, np.round(affected_kk*np.random.uniform(0.95,1.05,size=n_samples))).astype(int),
 'kasur_lipat_pcs': np.maximum(1, np.round(affected_kk*np.random.uniform(0.95,1.05,size=n_samples))).astype(int),
 'kompor_set': np.maximum(1, np.round(affected_kk*np.random.uniform(0.95,1.05,size=n_samples))).astype(int),
 'karpet_plastik_pcs': np.maximum(1, np.round(affected_kk*np.random.uniform(0.95,1.05,size=n_samples))).astype(int),
 'kipas_angin_pcs': np.maximum(1, np.round(affected_kk*0.7*(0.8+0.1*severity_level)*np.random.uniform(0.95,1.05,size=n_samples))).astype(int)})[TARGETS]

In [3]:
# Identical two-stage split: 40k/10k then fit 32k/val 8k/test 10k. Raw values for trees (no Normalization).
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=SEED)
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=SEED)
print(f'Fit {len(X_fit)} | validation {len(X_val)} | test {len(X_test)}')
x_fit, x_val, x_test = X_fit.to_numpy(np.float32), X_val.to_numpy(np.float32), X_test.to_numpy(np.float32)
y_fit_np, y_val_np, y_test_np = y_fit.to_numpy(np.float32), y_val.to_numpy(np.float32), y_test.to_numpy(np.float32)

Fit 32000 | validation 8000 | test 10000


In [4]:
# Model selection on VAL macro-MAE only; test touched once. HGB trains one estimator per target.
def macro_mae(yt, yp): return float(mean_absolute_error(yt, yp, multioutput='raw_values').mean())
val_mae = {}
hgb_grid = {'max_iter': [300, 500]}
best_hgb, best_hkey = None, None
for it in hgb_grid['max_iter']:
    m = MultiOutputRegressor(HistGradientBoostingRegressor(max_iter=it, early_stopping=True, random_state=SEED))
    m.fit(x_fit, y_fit_np)
    s = macro_mae(y_val_np, m.predict(x_val)); val_mae[f'hgb_{it}'] = s
    if best_hgb is None or s < val_mae[best_hkey]: best_hgb, best_hkey = m, f'hgb_{it}'
    print(f'HGB iter={it}: val macro-MAE={s:.4f}')
print('val grid:', val_mae)


HGB iter=300: val macro-MAE=2.4834
HGB iter=500: val macro-MAE=2.4834
val grid: {'hgb_300': 2.483391068530368, 'hgb_500': 2.483391068530368}


In [5]:
# Same metrics as DNN/DCN/MLP comparison, in original target units, test only.
def calculate_metrics(y_true, y_pred):
    mae_each = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
    rmse_each = np.sqrt(mean_squared_error(y_true, y_pred, multioutput='raw_values'))
    r2_each = r2_score(y_true, y_pred, multioutput='raw_values')
    return {'MAE_macro': float(np.mean(mae_each)), 'RMSE_macro': float(np.mean(rmse_each)),
        'R2_macro': float(np.mean(r2_each)),
        'MAE_weighted_all_values': float(mean_absolute_error(y_true, y_pred)),
        'RMSE_weighted_all_values': float(np.sqrt(mean_squared_error(y_true, y_pred)))}
pred_hgb = np.maximum(best_hgb.predict(x_test), 0.0)
summary = pd.DataFrame({'HGB_XGB_substitute': calculate_metrics(y_test_np, pred_hgb)}).T
per_item = pd.DataFrame(index=TARGETS)
per_item['HGB MAE'] = mean_absolute_error(y_test_np, pred_hgb, multioutput='raw_values')
summary.to_csv('logistics_hgb_comparison.csv')
per_item.to_csv('logistics_hgb_per_item_mae.csv')
pickle.dump(best_hgb, open('model_logistics_hgb.pkl','wb'))
summary  # paste beside DNN/DCN/MLP summary; keep in a separate thesis table with identical metric definitions


,MAE_macro,RMSE_macro,R2_macro,MAE_weighted_all_values,RMSE_weighted_all_values
HGB_XGB_substitute,2.505016,3.581237,0.998347,2.505016,4.513473
